# Assessment 1 Phase 2: Historical Airline Data Analysis with Apache Spark

## 1. Environment Setup

This section initialises the Apache Spark environment used for the analysis and confirms the Spark version available in the Docker-based Jupyter environment.

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Assessment1_Phase2") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.5


## 2. Dataset Loading and Initial Inspection

The approved U.S. airline on-time performance dataset is loaded into a Spark DataFrame. Initial checks are performed to confirm the number of records, number of columns, schema, and sample values before data cleaning and analysis.

In [11]:
file_path = "../T_ONTIME_REPORTING.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 539747
Number of columns: 36


In [12]:
df.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)


In [14]:
df.show(5, truncate=False)

+----+-------+-----+------------+-----------+--------------------+-----------------+--------+-----------------+-----------------+------+-----------------+----------------+---------------+----+-----------------+--------------+------------+--------+---------+-------------+---------+--------+-------+------------+--------+---------+-------------+---------+---------+--------+----------------+-------------------+--------+-------+--------+
|YEAR|QUARTER|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|FL_DATE             |OP_UNIQUE_CARRIER|TAIL_NUM|OP_CARRIER_FL_NUM|ORIGIN_AIRPORT_ID|ORIGIN|ORIGIN_CITY_NAME |ORIGIN_STATE_ABR|DEST_AIRPORT_ID|DEST|DEST_CITY_NAME   |DEST_STATE_ABR|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|DEP_DELAY_NEW|DEP_DEL15|TAXI_OUT|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|ARR_DELAY_NEW|ARR_DEL15|CANCELLED|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|FLIGHTS|DISTANCE|
+----+-------+-----+------------+-----------+--------------------+-----------------+--------+-----------------+---------------

## 3. Data Quality Assessment and Cleaning

The dataset is examined for missing values, incorrect data type, and records that may require cleaning before analytical queries are performed.

In [16]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show(truncate=False)

ConnectionRefusedError: [Errno 111] Connection refused

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Assessment1_Phase2") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.5


In [2]:
file_path = "../T_ONTIME_REPORTING.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 539747
Number of columns: 36


In [3]:
for c in df.columns:
    null_count = df.filter(F.col(c).isNull()).count()
    print(c, null_count)

YEAR 0
QUARTER 0
MONTH 0
DAY_OF_MONTH 0
DAY_OF_WEEK 0
FL_DATE 0
OP_UNIQUE_CARRIER 0
TAIL_NUM 2530
OP_CARRIER_FL_NUM 0
ORIGIN_AIRPORT_ID 0
ORIGIN 0
ORIGIN_CITY_NAME 0
ORIGIN_STATE_ABR 0
DEST_AIRPORT_ID 0
DEST 0
DEST_CITY_NAME 0
DEST_STATE_ABR 0
CRS_DEP_TIME 0
DEP_TIME 15886
DEP_DELAY 15923
DEP_DELAY_NEW 15923
DEP_DEL15 15923
TAXI_OUT 16227
TAXI_IN 16580
CRS_ARR_TIME 0
ARR_TIME 16580
ARR_DELAY 17478
ARR_DELAY_NEW 17478
ARR_DEL15 17478
CANCELLED 0
DIVERTED 0
CRS_ELAPSED_TIME 0
ACTUAL_ELAPSED_TIME 17478
AIR_TIME 17478
FLIGHTS 0
DISTANCE 0


In [4]:
df.groupBy("DIVERTED").count().show()

+--------+------+
|DIVERTED| count|
+--------+------+
|     0.0|538581|
|     1.0|  1166|
+--------+------+



In [5]:
df.groupBy("CANCELLED").count().show()

+---------+------+
|CANCELLED| count|
+---------+------+
|      0.0|523435|
|      1.0| 16312|
+---------+------+



### 3.1 Treatment of Cancelled and Diverted Flights

Missing values in arrival delay, air time, and actual elapsed time were investigated against cancellation and diversion indicators.The combined number of cancelled and diverted flights corresponded with the number of missing arrival-related observations. Therefore, these missing values were treated as operationally meaningful rather than as random data-quality errors. For analyses of completed-flight delay performance, cancelled and diverted flights are excluded while the original dataset is retained for analyses involving cancellation or diversion behaviour.

In [6]:
df_completed = df.filter(
    (F.col("CANCELLED") == 0) &
    (F.col("DIVERTED") == 0)
)

print("Completed flights:", df_completed.count())

Completed flights: 522269


### 3.2 Date Conversion

The flight date field is converted from string format to a Spark date type to support reliable time-based analysis and later feature creation.

In [7]:
df_completed = df_completed.withColumn(
    "FL_DATE",
    F.to_date(F.col("FL_DATE"), "M/d/yyyy h:mm:ss a")
)

df_completed.select("FL_DATE").show(5)

+----------+
|   FL_DATE|
+----------+
|2025-01-01|
|2025-01-01|
|2025-01-01|
|2025-01-01|
|2025-01-01|
+----------+
only showing top 5 rows



In [8]:
df_completed.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: date (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |

### 3.3 Derived Route Feature

A route identifier is created by combining the origin and destination airport codes. This supports route-level delay comparisons in later analysis.

In [9]:
df_completed = df_completed.withColumn(
    "ROUTE",
    F.concat_ws("-", F.col("ORIGIN"), F.col("DEST"))
)

df_completed.select("ORIGIN", "DEST", "ROUTE").show(5)

+------+----+-------+
|ORIGIN|DEST|  ROUTE|
+------+----+-------+
|   SFO| JFK|SFO-JFK|
|   JFK| SFO|JFK-SFO|
|   SAT| CLT|SAT-CLT|
|   JFK| LAX|JFK-LAX|
|   BOS| LAX|BOS-LAX|
+------+----+-------+
only showing top 5 rows



### 3.4 Scheduled Departure Hour

The sheduled departure time is converted into an hour-of-day feature to support analysis of delay patterns across different departure periods. 

In [10]:
df_completed = df_completed.withColumn(
    "DEP_HOUR",
    F.floor(F.col("CRS_DEP_TIME") / 100)
)

df_completed.select("CRS_DEP_TIME", "DEP_HOUR").show(10)

+------------+--------+
|CRS_DEP_TIME|DEP_HOUR|
+------------+--------+
|        1030|      10|
|         600|       6|
|         819|       8|
|        2100|      21|
|         801|       8|
|        1130|      11|
|        1104|      11|
|        1259|      12|
|         746|       7|
|        2025|      20|
+------------+--------+
only showing top 10 rows



### 3.5 Departure Time Period

Sheduled departure hours are grouped into broad time-of-day categories to support comparison of delay behaviour across different periods of the day.

In [14]:
df_completed = df_completed.withColumn(
    "DEP_PERIOD",
    F.when(F.col("DEP_HOUR") < 6, "Night")
     .when(F.col("DEP_HOUR") < 12, "Morning")
     .when(F.col("DEP_HOUR") < 18, "Afternoon")
     .otherwise("Evening")
)

df_completed.select("DEP_HOUR", "DEP_PERIOD").show(10)

+--------+----------+
|DEP_HOUR|DEP_PERIOD|
+--------+----------+
|      10|   Morning|
|       6|   Morning|
|       8|   Morning|
|      21|   Evening|
|       8|   Morning|
|      11|   Morning|
|      11|   Morning|
|      12| Afternoon|
|       7|   Morning|
|      20|   Evening|
+--------+----------+
only showing top 10 rows



## 4. Initial Delay Analysis

This section examines departure and arrival delay patterns for completed flights. Average delays are compared across carriers, routes, and departure periods to identify operational patterns in the sataset

In [21]:
carrier_delay = (
    df_completed
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(
        F.round(F.avg("DEP_DELAY"), 2).alias("AVG_DEP_DELAY"),
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .orderBy(F.desc("AVG_ARR_DELAY"))
)

carrier_delay.show()


+-----------------+-------------+-------------+------------+
|OP_UNIQUE_CARRIER|AVG_DEP_DELAY|AVG_ARR_DELAY|FLIGHT_COUNT|
+-----------------+-------------+-------------+------------+
|               OH|        17.57|        14.14|       19151|
|               F9|        14.43|        10.82|       15110|
|               G4|        15.48|         9.54|        9206|
|               OO|         13.0|         7.94|       63502|
|               HA|         6.82|         6.04|        6540|
|               B6|        11.11|         5.81|       17558|
|               AA|        12.86|         5.57|       72082|
|               DL|        12.71|         5.22|       74025|
|               MQ|         8.92|         4.81|       20831|
|               UA|         8.29|         1.29|       60668|
|               NK|         7.36|         0.65|       16946|
|               AS|         5.15|        -0.12|       17847|
|               WN|         7.26|        -1.17|      102120|
|               YX|     

The results show clear variation in average delay performance across carriers. OH recorded the highest average arrival delay, while YX recorded a negative average arrival delay, indicating that its flights arrived slightly earlier than scheduled on average. Flight counts also vary considerably across carriers, so delay comparisons should be interpreted together with carrier traffic volume.

### 4.1 Average Delay by Route

Average departure and arrival delays are calculated for each origin-destination route. Flight counts are included so that delay results can be interpreted together with route traffic volume. 

In [22]:
route_delay = (
    df_completed
    .groupBy("ROUTE")
    .agg(
        F.round(F.avg("DEP_DELAY"), 2).alias("AVG_DEP_DELAY"),
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .filter(F.col("FLIGHT_COUNT") >= 500)
    .orderBy(F.desc("AVG_ARR_DELAY"))
)

route_delay.show(20, truncate=False)

+-------+-------------+-------------+------------+
|ROUTE  |AVG_DEP_DELAY|AVG_ARR_DELAY|FLIGHT_COUNT|
+-------+-------------+-------------+------------+
|ATL-MIA|22.03        |20.55        |521         |
|ATL-MCO|19.83        |18.45        |664         |
|FLL-ATL|23.98        |17.72        |554         |
|MIA-ATL|22.96        |16.64        |522         |
|ATL-FLL|16.53        |15.11        |552         |
|MCO-ATL|19.28        |12.53        |651         |
|DCA-ATL|19.65        |11.95        |527         |
|ATL-LGA|18.06        |11.46        |553         |
|LGA-MIA|15.92        |9.79         |567         |
|LGA-ATL|15.69        |9.74         |551         |
|ATL-DCA|12.57        |9.64         |531         |
|LGA-DFW|10.7         |8.71         |520         |
|MIA-LGA|17.02        |7.79         |563         |
|SAN-LAS|13.01        |7.66         |573         |
|SJU-MCO|17.42        |7.28         |526         |
|LAS-SAN|12.38        |6.88         |576         |
|SFO-LAS|7.99         |5.86    

Among routes with at least 500 completed flights, ATL-MIA recorded the highest average arrival delay at 20.55 minutes. Several other high-delay routes were also associated with ATL and MIA. Applying a minimum flight-count threshold helps ensure that the comparison is based on routes with sufficient traffic volume rather than very small samples.

### 4.2 Average Delay by Departure Period

Average departure and arrival delays are compared across broad time-of-day categories to identify whether delay performance varies by sheduled departure period.

In [25]:
period_delay = (
    df_completed
    .groupBy("DEP_PERIOD")
    .agg(
        F.round(F.avg("DEP_DELAY"), 2).alias("AVG_DEP_DELAY"),
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .orderBy(F.desc("AVG_ARR_DELAY"))
)
                      
period_delay.show()

+----------+-------------+-------------+------------+
|DEP_PERIOD|AVG_DEP_DELAY|AVG_ARR_DELAY|FLIGHT_COUNT|
+----------+-------------+-------------+------------+
|   Evening|        13.85|         6.88|      108886|
| Afternoon|        12.09|         5.69|      191227|
|     Night|         7.98|         1.27|       14430|
|   Morning|          6.8|         0.52|      207726|
+----------+-------------+-------------+------------+



The results indicate that delay performance worsens later in the day. Morning flights recorded the lowest average arrival delay, while evening flights recorded the highest. This pattern is consistent with delays accumulating throughout the operating day, although the descriptive analysis does not establish causality.

## 5. Advanced Spark Analysis Using Window Functions

Window functions are used to rank routes within each carrier according to average arrival delay. A minimum flight-count threshold is applied to reduce the influence of routes with very small numbers of observations.

In [26]:
carrier_route_delay = (
    df_completed
    .groupBy("OP_UNIQUE_CARRIER", "ROUTE")
    .agg(
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .filter(F.col("FLIGHT_COUNT") >= 100)
)

carrier_route_delay.show(10, truncate=False)

+-----------------+-------+-------------+------------+
|OP_UNIQUE_CARRIER|ROUTE  |AVG_ARR_DELAY|FLIGHT_COUNT|
+-----------------+-------+-------------+------------+
|AA               |MCO-PHL|5.47         |203         |
|AA               |ORD-MIA|2.23         |252         |
|DL               |SFO-SLC|5.49         |134         |
|DL               |SLC-SAN|-1.43        |150         |
|DL               |ATL-PIT|12.37        |172         |
|OO               |ASE-DEN|10.76        |276         |
|WN               |SAT-LAS|-1.09        |109         |
|AA               |TPA-CLT|-1.23        |305         |
|AA               |DFW-DTW|10.02        |141         |
|B6               |FLL-LAX|0.95         |130         |
+-----------------+-------+-------------+------------+
only showing top 10 rows



In [29]:
carrier_window = Window.partitionBy(
    "OP_UNIQUE_CARRIER"
).orderBy(
    F.desc("AVG_ARR_DELAY")
)

In [31]:
ranked_routes = carrier_route_delay.withColumn(
    "DELAY_RANK",
    F.row_number().over(carrier_window)
)

ranked_routes.filter(
    F.col("DELAY_RANK") <= 3
).orderBy(
    "OP_UNIQUE_CARRIER",
    "DELAY_RANK"
).show(50, truncate=False)

+-----------------+-------+-------------+------------+----------+
|OP_UNIQUE_CARRIER|ROUTE  |AVG_ARR_DELAY|FLIGHT_COUNT|DELAY_RANK|
+-----------------+-------+-------------+------------+----------+
|AA               |EGE-DFW|34.86        |110         |1         |
|AA               |DFW-MFE|31.27        |171         |2         |
|AA               |MFE-DFW|31.19        |170         |3         |
|AS               |SEA-ANC|11.58        |396         |1         |
|AS               |GEG-SEA|11.2         |119         |2         |
|AS               |SLC-SEA|11.09        |102         |3         |
|B6               |SJU-BOS|25.88        |120         |1         |
|B6               |BOS-PBI|21.7         |188         |2         |
|B6               |BOS-TPA|20.91        |123         |3         |
|DL               |MIA-ATL|25.22        |274         |1         |
|DL               |ATL-IAD|24.74        |144         |2         |
|DL               |FLL-ATL|23.07        |357         |3         |
|F9       

The window-function analysis identified the highest-delay routes within each carrier after restricting the comparison to routes with at least 100 completed flights. Considerable variation was observed both between carriers and between routes operated by the same carrier. The use of a partitioned window allowed routes to be ranked independently within each carrier, providing a more detailed comparison than carrier-level averages alone.